## Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

# Important libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from os.path import join, exists
import joblib
import sys
import torch
from os import rename
sys.path.append("../../")

from src.configs.default_configs import fn_model, fn_pred, fn_pred_perf, device
from src.configs.crc_config import data_name, postnet_param
from src.file_manager.filepath import FilePath
from src.models.postnet.posterior_networks.run import run
from src.models.postnet.posterior_networks.run_eval import run_eval
from src.misc import set_seed_pytorch
from src.data_processing.dataloader import get_tabular_dl_dict_postnet
from src.models.postnet.result_processing import process_pn_results
from src.file_manager.load_save_df import load_pred_df, save_pred_perf_df
from src.models.postnet.posterior_networks.PosteriorNetwork import PosteriorNetwork
from src.evaluation.evaluate import get_model_performance
from seed_file import seed
seed = 1

batch_size = 32
eval_batch_size = 128

tuning_seed = 2024

fp = FilePath(data_name=data_name, seed=seed)
fp_scaler_file = join(fp.get_preprocessed_folder(), f"minmax_scaler.pickle")
fp_encoder_file = join(fp.get_preprocessed_folder(), f"encoder.pickle")
fp_split_dict_file = join(fp.get_preprocessed_folder(), f"split_dict.joblib")
fp_split_dict_oversampled_file = join(fp.get_preprocessed_folder(), f"split_dict_oversampled.joblib")

directory_model=fp.get_parent_folder(folder_name=fn_model)
directory_results=fp.get_parent_folder(folder_name=fn_pred)
fn_cur_model = f"model-dpn-{seed}-{data_name}-{seed}-{postnet_param['architecture']}-{postnet_param['input_dims']}-{postnet_param['output_dim']}"

# Load Data

In [ ]:
split_dict_scaled = joblib.load(fp_split_dict_file)
split_dict_oversampled = joblib.load(fp_split_dict_oversampled_file)
feat_cols_w_pc = ['Age_interview', 'PC1', 'PC2', 'PC3', 'BMI', 'telomere length', 'aHEI2010score', 'aMED', 'DASH', 'SBP', 'DBP', 'Leisure screen time', 'z_pgs000055', 'z_pgs000734', 'Sex (0=Male, 1=Female)', 'alcohol_DailyandWeekly(1)vsMonthlyandNonDrinkers(0)', 'smoke_ex(1)', 'smoke_current(2)', 'Prevalent_diabetes']
target_col = "colorectal cancer"
split_dict_scaled["feat_cols"] = feat_cols_w_pc
split_dict_scaled["target_col"] = target_col 
split_dict_scaled["num_classes"] = 2
split_dict_oversampled["feat_cols"] = feat_cols_w_pc
split_dict_oversampled["target_col"] = target_col 
split_dict_oversampled["num_classes"] = 2

# Train

In [ ]:
if not exists(join(directory_model, fn_cur_model)):
    results_metrics = run(
        data_dict=split_dict_oversampled,
        # Directory
        directory_model=directory_model,
        directory_results=directory_results,
        # Seeds
        seed_dataset=seed, # No shuffling
        seed_model=seed,
        dl_func=get_tabular_dl_dict_postnet,
        **postnet_param
    )

# Prediction

In [ ]:
ood_postnet_param = postnet_param.copy()
fp_results = run_eval(
    data_dict=split_dict_scaled,
    ood_data_dicts = {},
    # Directory
    directory_model=directory_model,
    directory_results=directory_results,
    # Seeds
    seed_dataset=seed, # No shuffling
    seed_model=seed,
    **ood_postnet_param,
    dl_func=get_tabular_dl_dict_postnet,
    fn_model=fn_cur_model,
)

# Process Output

In [ ]:
process_pn_results(
    split_dict_oversampled, {}, directory_results, fp, only_test=True
)

# Evaluate Pred Perf

In [ ]:
pred_df = load_pred_df(fp=fp, ModelClass=PosteriorNetwork)
perf_df = get_model_performance(
    all_pred_df=pred_df, data_dict=split_dict_scaled, label="pn", perf_split_col="split")
save_pred_perf_df(pred_perf_df=perf_df, fp=fp, ModelClass=PosteriorNetwork)
perf_df